# MC3 model test on a free GPU

This tests the model half of the MC3 solution (reading images, answering, citing) on a free NVIDIA GPU, **before** the AMD GPU is available.
It uses the same code as the real container, with a smaller model from the same family (Qwen3-VL-4B instead of 8B).
It does **not** test the AMD/ROCm Docker image, only whether the prompt, the image reading and the answer rules work.

**Before you start:** in the top menu choose **Runtime > Change runtime type > T4 GPU**, then run the cells one at a time, top to bottom.
Total time is about 15 to 30 minutes. Cell 6 is the slow one (it loads the model).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Upload the kit
Click **Choose files** and pick `mc3-starter-kit-with-reader.zip`.

In [ ]:
from google.colab import files
up = files.upload()
!unzip -q -o mc3-starter-kit-with-reader.zip
%cd /content/mc3-starter-kit
!ls

## 3. Install the packages
Only runs in a separate process later, so no restart is needed.

In [ ]:
!pip install -q -U "transformers>=4.57" pypdf huggingface_hub

## 4. Prepare the sample corpus and download the model (about 9 GB)

In [ ]:
!sh setup.sh
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-VL-4B-Instruct", local_dir="/content/model",
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"])
print("model downloaded")

## 5. Settings
`float16` because the free T4 GPU has no bfloat16. `MC3_DEBUG` prints the model's raw replies.

In [ ]:
import os
os.environ.update({
    "MC3_MODEL_DIR": "/content/model", "MC3_DTYPE": "float16", "MC3_LLM_SOCKET": "/tmp/mc3.sock",
    "MC3_INDEX_DIR": "/content/idx", "MC2_OUTPUT_DIR": "/content/out", "MC3_DEBUG": "1",
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
})
print("settings ready")

## 6. Index pass (loads the model, reads the images with it)
This is the slow cell. It should finish in a few minutes and print a summary of skipped files.

In [ ]:
!python3 starter/app/app.py --index mc3-corpus
!echo "---- model server log ----"
!tail -n 25 /tmp/mc3_model_server.log

## 7. What did the model read from the two images?
The pin diagram and the asset label. The text below should contain the pin name and the board revision.

In [ ]:
import pickle
idx = pickle.load(open("/content/idx/index.pkl", "rb"))
for d in idx["docs"]:
    if d["kind"] == "image":
        print(d["file"]); print(d["text"][:700]); print("-" * 50)

## 8. Run the 10 sample questions
One fresh process per question, like the real grader. Each line shows PASS or FAIL, the time, the answer, and whether the citations are right. For failures it also prints the model's raw reply.

In [ ]:
!python3 score_samples.py --local mc3-corpus

## 9. Clean up
Stops the model server so the GPU is freed.

In [ ]:
!pkill -f "[m]odel_server.py" 

## What to send back
Copy the text output (or screenshots) of **cells 6, 7 and 8** and send them to me. Errors are useful too, so send those as well.
Note: Colab runs as the admin user, so the "unreadable file" case is not really tested here. That is fine for this test.